# Stage 15 — forensic risk scoring

Build a transparent SIM-level risk score from independent evidence layers. The score is designed to prioritize investigation, not replace the ground-truth label or the machine-learning probability.

Traffic and social evidence receive the highest weights. Temporal, mobility, and adaptive deviation evidence provide supporting signals. Each indicator is converted to a percentile relative to known legitimate SIMs and capped before aggregation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE=Path('../outputs'); OUT=BASE/'risk_scoring/all_naive_12pct_200sims'; OUT.mkdir(parents=True,exist_ok=True)
traffic=pd.read_csv(BASE/'traffic/all_naive_12pct_200sims/traffic_features.csv',dtype={'sim_number':'string'})
temporal=pd.read_csv(BASE/'temporal/all_naive_12pct_200sims/temporal_features.csv',dtype={'sim_number':'string'})
mobility=pd.read_csv(BASE/'mobility/all_naive_12pct_200sims/mobility_features.csv',dtype={'sim_number':'string'})
social=pd.read_csv(BASE/'social/all_naive_12pct_200sims/social_network_features.csv',dtype={'sim':'string'}).rename(columns={'sim':'sim_number'})
deviation=pd.read_csv(BASE/'deviation/all_naive_12pct_200sims/sim_behavioral_deviation_summary.csv',dtype={'sim':'string'}).rename(columns={'sim':'sim_number'})
frames=[traffic,temporal,mobility,social,deviation]
profile=frames[0]
for frame in frames[1:]: profile=profile.merge(frame,on=['sim_number','known_fraud'] if 'known_fraud' in frame.columns else ['sim_number'],how='left',suffixes=('','_extra'))
profile=profile.loc[:,~profile.columns.duplicated()].copy()
print('profiles:',len(profile),'fraud labels:',int(profile.known_fraud.sum()))

In [ ]:
# A feature contributes evidence only when it is high relative to the legitimate reference group.
indicators={
 'traffic_activity':('activity_count',.28),
 'traffic_duration':('total_duration',.18),
 'traffic_contacts':('unique_contacts',.14),
 'traffic_outgoing':('outgoing_share',.12),
 'social_degree':('outgoing_degree',.10),
 'temporal_hours':('active_hours',.06),
 'temporal_night':('night_activity_ratio',.04),
 'mobility_changes':('observed_cell_changes',.03),
 'adaptive_evidence':('max_adaptive_evidence',.05),
}
legit=profile.loc[~profile.known_fraud]
contributions=[]
for name,(column,weight) in indicators.items():
    reference=legit[column].dropna()
    q1=reference.quantile(.25); q3=reference.quantile(.75); iqr=q3-q1
    if pd.isna(iqr) or iqr==0: iqr=reference.std() if reference.std()>0 else 1.0
    # Robust evidence score: 0 at/below Q3, increasing above Q3, capped at 1.
    raw=(profile[column]-q3)/iqr
    component=raw.clip(lower=0,upper=4)/4
    profile[name+'_component']=component.fillna(0)
    profile[name+'_flag']=profile[column]>q3
    profile[name+'_weight']=weight
    contributions.append({'indicator':name,'source_column':column,'weight':weight,'legitimate_q3':q3,'legitimate_iqr':iqr,'flagged_users':int(profile[name+'_flag'].sum())})
profile['risk_score']=sum(profile[name+'_component']*weight for name,(_,weight) in indicators.items())*100
profile['evidence_layer_count']=(profile[[name+'_flag' for name in indicators]]).sum(axis=1)
profile['high_priority']= (profile.risk_score>=50) & (profile.evidence_layer_count>=3)
profile=profile.sort_values(['risk_score','evidence_layer_count'],ascending=False)
profile.to_csv(OUT/'forensic_risk_scores.csv',index=False)
pd.DataFrame(contributions).to_csv(OUT/'risk_indicator_reference.csv',index=False)
print(profile[['sim_number','risk_score','evidence_layer_count','high_priority','known_fraud']].head(20))

In [ ]:
# Evaluate the investigative ranking against known labels without using labels to build the score.
ranked=profile.reset_index(drop=True)
rows=[]
for k in [10,25,50,100,200]:
    top=ranked.head(min(k,len(ranked))); rows.append({'top_k':len(top),'fraud_found':int(top.known_fraud.sum()),'precision_at_k':float(top.known_fraud.mean()),'recall_at_k':float(top.known_fraud.sum()/ranked.known_fraud.sum())})
ranking=pd.DataFrame(rows); ranking.to_csv(OUT/'risk_score_top_k_metrics.csv',index=False)
conf=pd.crosstab(profile.known_fraud,profile.high_priority).reindex(index=[False,True],columns=[False,True],fill_value=0); conf.index=['known_legitimate','known_fraudulent']; conf.columns=['not_high_priority','high_priority']; conf.to_csv(OUT/'risk_score_priority_table.csv')
print(ranking); print(conf)

## Interpretation

The risk score ranks SIMs for investigation using multiple capped evidence contributions. A high score indicates unusual behavior across the selected indicators. It does not establish fraud by itself. The top-k table is the main operational evaluation because analysts typically investigate a limited shortlist.